# 08 — Compliance: GDPR, Privacy & Responsible AI

## Part 1: Theory

### 1.1 AI Governance Landscape

| Regulation | Region | Focus | AI Impact |
|-----------|--------|-------|----------|
| **GDPR** | EU | Personal data protection | Training data, user queries, right to erasure |
| **EU AI Act** | EU | AI risk classification | High-risk AI needs conformity assessment |
| **CCPA/CPRA** | California | Consumer privacy | Opt-out of data sale, deletion rights |
| **HIPAA** | US | Health data | PHI in medical AI systems |
| **SOC 2** | Global | Security controls | Audit trails, access controls |

### 1.2 GDPR Core Principles (Article 5)

1. **Lawfulness, fairness, transparency** — Tell users what you do with their data
2. **Purpose limitation** — Collect data only for specified purposes
3. **Data minimization** — Only collect what's necessary
4. **Accuracy** — Keep data correct and up-to-date
5. **Storage limitation** — Don't keep data longer than needed
6. **Integrity & confidentiality** — Protect data with security measures
7. **Accountability** — Demonstrate compliance

### 1.3 Data Subject Rights

| Right | Article | What It Means for AI |
|-------|---------|---------------------|
| Access | 15 | User can request all data you hold about them |
| Rectification | 16 | User can correct inaccurate data |
| Erasure | 17 | "Right to be forgotten" — delete all user data |
| Portability | 20 | Export data in machine-readable format |
| Object | 21 | User can opt out of automated processing |
| Explanation | 22 | Right to understand automated decisions |

### 1.4 EU AI Act Risk Levels

```
UNACCEPTABLE (banned):     Social scoring, real-time biometric surveillance
HIGH-RISK (regulated):     Hiring AI, credit scoring, medical diagnosis
LIMITED RISK (transparency): Chatbots (must disclose AI), deepfakes
MINIMAL RISK (free):       Spam filters, game AI, recommendations
```

### 1.5 Privacy by Design (7 Principles)

1. Proactive not reactive
2. Privacy as the default
3. Privacy embedded into design
4. Full functionality (no trade-offs)
5. End-to-end security
6. Visibility and transparency
7. Respect for user privacy

### 1.6 AI-Specific Challenges

- **Model memorization**: LLMs can regurgitate training data (PII leakage)
- **Inference attacks**: Extracting training data from model outputs
- **Right to explanation**: How do you explain a 70B parameter model's decision?
- **Training data rights**: Was consent obtained for all training data?

---

## Part 2: Implementation

In [ ]:
import json
import hashlib
import re
from datetime import datetime, timedelta
from cryptography.fernet import Fernet

### PII Detection (Regex-based, no external deps needed)

In [ ]:
class PIIDetector:
    """Detect PII using regex patterns."""
    PATTERNS = {
        "EMAIL": r'[a-zA-Z0-9._%+-]+@[a-zA-Z0-9.-]+\.[a-zA-Z]{2,}',
        "PHONE": r'\+?1?[-.\s]?\(?\d{3}\)?[-.\s]?\d{3}[-.\s]?\d{4}',
        "SSN": r'\d{3}-\d{2}-\d{4}',
        "CREDIT_CARD": r'\d{4}[- ]?\d{4}[- ]?\d{4}[- ]?\d{4}',
        "IP_ADDRESS": r'\d{1,3}\.\d{1,3}\.\d{1,3}\.\d{1,3}',
    }
    
    def detect(self, text):
        findings = []
        for entity_type, pattern in self.PATTERNS.items():
            for match in re.finditer(pattern, text):
                findings.append({"type": entity_type, "value": match.group(), "start": match.start(), "end": match.end()})
        return findings
    
    def has_pii(self, text):
        return len(self.detect(text)) > 0

detector = PIIDetector()

test_texts = [
    "My email is john.smith@company.com and phone is +1-555-123-4567",
    "SSN: 123-45-6789, card: 4111-1111-1111-1111",
    "What is the refund policy?",  # No PII
]

for text in test_texts:
    findings = detector.detect(text)
    print(f"Text: {text[:50]}...")
    print(f"  PII found: {len(findings)} → {[f['type'] for f in findings]}\n")

### PII Anonymization

In [ ]:
class PIIAnonymizer:
    def __init__(self):
        self.detector = PIIDetector()
    
    def anonymize(self, text, method="replace"):
        findings = self.detector.detect(text)
        # Sort by position (reverse) to replace without offset issues
        for f in sorted(findings, key=lambda x: x["start"], reverse=True):
            placeholder = f"<{f['type']}>"
            if method == "hash":
                placeholder = hashlib.sha256(f["value"].encode()).hexdigest()[:8]
            elif method == "mask":
                placeholder = f["value"][0] + "*" * (len(f["value"]) - 2) + f["value"][-1]
            text = text[:f["start"]] + placeholder + text[f["end"]:]
        return text

anonymizer = PIIAnonymizer()
text = "Contact john@example.com or call 555-123-4567 about card 4111-1111-1111-1111"

print(f"Original:  {text}")
print(f"Replace:   {anonymizer.anonymize(text, 'replace')}")
print(f"Hash:      {anonymizer.anonymize(text, 'hash')}")
print(f"Mask:      {anonymizer.anonymize(text, 'mask')}")

### Privacy-Preserving RAG Pipeline

In [ ]:
class PrivacyRAG:
    def __init__(self):
        self.anonymizer = PIIAnonymizer()
        self.audit_log = []
    
    def process(self, query, user_id):
        # 1. Detect & anonymize PII before sending to LLM
        clean_query = self.anonymizer.anonymize(query)
        pii_found = self.anonymizer.detector.has_pii(query)
        
        # 2. Audit log (hashed user ID, no PII stored)
        self.audit_log.append({
            "timestamp": datetime.now().isoformat(),
            "user_hash": hashlib.sha256(user_id.encode()).hexdigest()[:12],
            "pii_detected": pii_found,
            "query_length": len(query),
        })
        
        # 3. Send clean query to LLM (simulated)
        answer = f"[Response to: {clean_query[:50]}]"
        return {"answer": answer, "pii_scrubbed": pii_found, "clean_query": clean_query}

rag = PrivacyRAG()
result = rag.process("My email is alice@corp.com. What's the refund policy?", "user-123")
print(f"Clean query: {result['clean_query']}")
print(f"PII scrubbed: {result['pii_scrubbed']}")
print(f"Audit log: {json.dumps(rag.audit_log, indent=2)}")

### Data Retention & Right to Erasure

In [ ]:
class DataRetention:
    def __init__(self, retention_days=30):
        self.retention_days = retention_days
        self.store = {}  # user_id -> records
    
    def store_interaction(self, user_id, data):
        record = {**data, "created": datetime.now().isoformat(),
                  "expires": (datetime.now() + timedelta(days=self.retention_days)).isoformat()}
        self.store.setdefault(user_id, []).append(record)
    
    def delete_user(self, user_id):  # Article 17
        if user_id in self.store:
            count = len(self.store.pop(user_id))
            return {"deleted": True, "records_removed": count}
        return {"deleted": False, "reason": "not_found"}
    
    def export_user(self, user_id):  # Article 20
        return json.dumps(self.store.get(user_id, []), indent=2)
    
    def purge_expired(self):
        now = datetime.now()
        purged = 0
        for uid in list(self.store.keys()):
            before = len(self.store[uid])
            self.store[uid] = [r for r in self.store[uid] if datetime.fromisoformat(r["expires"]) > now]
            purged += before - len(self.store[uid])
            if not self.store[uid]: del self.store[uid]
        return purged

retention = DataRetention(retention_days=30)
retention.store_interaction("user-1", {"query": "refund policy", "response_len": 150})
retention.store_interaction("user-1", {"query": "cancel account", "response_len": 80})
print(f"Export: {retention.export_user('user-1')}")
print(f"Delete: {retention.delete_user('user-1')}")

### Encryption at Rest

In [ ]:
class EncryptedStore:
    def __init__(self):
        self.key = Fernet.generate_key()
        self.cipher = Fernet(self.key)
        self.store = {}
    
    def put(self, key, plaintext):
        self.store[key] = self.cipher.encrypt(plaintext.encode())
    
    def get(self, key):
        return self.cipher.decrypt(self.store[key]).decode() if key in self.store else None

enc = EncryptedStore()
enc.put("query-001", "What is my balance for card 4111-1111-1111-1111?")
print(f"Encrypted: {self.store['query-001'][:40]}..." if False else f"Encrypted: {enc.store['query-001'][:40]}...")
print(f"Decrypted: {enc.get('query-001')}")

## Part 3: Compliance Checklist

| GDPR Article | Requirement | Our Implementation |
|---|---|---|
| Art. 5 | Data minimization | PII scrubbing before LLM |
| Art. 13 | Transparency | Disclose AI processing to users |
| Art. 17 | Right to erasure | `delete_user()` |
| Art. 20 | Data portability | `export_user()` JSON |
| Art. 25 | Privacy by design | Anonymize → then process |
| Art. 30 | Processing records | Audit log with hashed IDs |
| Art. 32 | Security | Encryption at rest |
| Art. 35 | DPIA | Document risks before deployment |

### Key Takeaways

1. **PII detection BEFORE LLM** — never send personal data to external APIs
2. **Anonymize, don't just delete** — preserve utility while removing identity
3. **Audit everything** — but audit with hashed IDs, not raw PII
4. **Retention policies** — auto-purge is safer than manual deletion
5. **Encrypt at rest** — defense in depth
6. **Right to erasure** must be implementable in <72 hours (GDPR requirement)

### Next → 09_prompt_engineering.ipynb